# 5.2

## Imports and configuration

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import make_grid
from torchvision.models import inception_v3, Inception_V3_Weights
import matplotlib.pyplot as plt
import numpy as np
from scipy import linalg
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

Z_DIM = 100
BATCH_SIZE = 128
EPOCHS = 3
LR = 0.0002
BETA1 = 0.5

## Data preparation (CIFAR-10)

In [8]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

## Model architecture (DCGAN)

In [9]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

class Generator(nn.Module):
    def __init__(self, z_dim):
        super(Generator, self).__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 128, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).view(-1, 1)

netG = Generator(Z_DIM).to(device)
netD = Discriminator().to(device)
netG.apply(weights_init)
netD.apply(weights_init)

Discriminator(
  (net): Sequential(
    (0): Conv2d(3, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (1): LeakyReLU(negative_slope=0.2, inplace=True)
    (2): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): LeakyReLU(negative_slope=0.2, inplace=True)
    (5): Conv2d(256, 512, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (6): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): LeakyReLU(negative_slope=0.2, inplace=True)
    (8): Conv2d(512, 1, kernel_size=(4, 4), stride=(1, 1), bias=False)
    (9): Sigmoid()
  )
)

## Support functions and FID metric

In [10]:
inception_model = inception_v3(weights=Inception_V3_Weights.DEFAULT, transform_input=False).to(device)
inception_model.fc = nn.Identity()
inception_model.eval()

def get_inception_features(images):
    images = torch.nn.functional.interpolate(images, size=(299, 299), mode='bilinear', align_corners=False)
    with torch.no_grad():
        features = inception_model(images)
    return features.cpu().numpy()

def calculate_fid(real_features, fake_features):
    mu1, sigma1 = real_features.mean(axis=0), np.cov(real_features, rowvar=False)
    mu2, sigma2 = fake_features.mean(axis=0), np.cov(fake_features, rowvar=False)
    
    ssdiff = np.sum((mu1 - mu2)**2.0)
    covmean = linalg.sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
        
    fid = ssdiff + np.trace(sigma1 + sigma2 - 2.0 * covmean)
    return fid

## Main learning loop

In [13]:
TEST_BATCH_LIMIT = 5

criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=LR, betas=(BETA1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=LR, betas=(BETA1, 0.999))

fixed_noise = torch.randn(64, Z_DIM, 1, 1, device=device)

G_losses = []
D_losses = []
fid_scores = []
img_list = []

for epoch in range(EPOCHS):
    epoch_g_loss = 0.0
    epoch_d_loss = 0.0
    
    for i, data in enumerate(dataloader, 0):
        # only for tests
        # if i >= TEST_BATCH_LIMIT:
        #     break
        # Update D
        netD.zero_grad()
        real_cpu = data[0].to(device)
        b_size = real_cpu.size(0)
        
        # Label smoothing dla prawdziwych obrazów (0.9 zamiast 1.0)
        label_real = torch.full((b_size, 1), 0.9, dtype=torch.float, device=device)
        output_real = netD(real_cpu)
        errD_real = criterion(output_real, label_real)
        errD_real.backward()

        noise = torch.randn(b_size, Z_DIM, 1, 1, device=device)
        fake = netG(noise)
        label_fake = torch.full((b_size, 1), 0.0, dtype=torch.float, device=device)
        
        output_fake = netD(fake.detach())
        errD_fake = criterion(output_fake, label_fake)
        errD_fake.backward()
        optimizerD.step()

        # Update G
        netG.zero_grad()
        label_g = torch.full((b_size, 1), 1.0, dtype=torch.float, device=device)
        output_g = netD(fake)
        errG = criterion(output_g, label_g)
        errG.backward()
        optimizerG.step()
        
        epoch_d_loss += (errD_real.item() + errD_fake.item())
        epoch_g_loss += errG.item()

    G_losses.append(epoch_g_loss / len(dataloader))
    D_losses.append(epoch_d_loss / len(dataloader))

    # Obliczanie FID na podzbiorze (dla wydajności) na koniec epoki
    if (epoch + 1) % 5 == 0:
        real_feats = get_inception_features(real_cpu[:64])
        fake_feats = get_inception_features(fake[:64])
        fid = calculate_fid(real_feats, fake_feats)
        fid_scores.append(fid)
        print(f'Epoch [{epoch+1}/{EPOCHS}] | Loss_D: {D_losses[-1]:.4f} | Loss_G: {G_losses[-1]:.4f} | FID: {fid:.2f}')
    
    with torch.no_grad():
        fake_display = netG(fixed_noise).detach().cpu()
    img_list.append(make_grid(fake_display, padding=2, normalize=True))

RuntimeError: [enforce fail at alloc_cpu.cpp:117] data. DefaultCPUAllocator: not enough memory: you tried to allocate 8388608 bytes.

## Data visualization

In [ ]:
plt.figure(figsize=(10, 5))
plt.title("Generator and Discriminator Loss During Training")
plt.plot(G_losses, label="G")
plt.plot(D_losses, label="D")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

# Wizualizacja siatki wygenerowanych obrazów z ostatniej epoki
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("Synthetic Images")
plt.imshow(np.transpose(img_list[-1], (1, 2, 0)))
plt.show()